# AlphaFold Active Labeling Inference

Cleaned batch odds-ratio workflow using the compact merged PTM/IDR table. Original exploratory notebooks are preserved in `archive/`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name in {'Stance', 'Alphafold', 'CheXpert', 'BRCA'} else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    binary_odds_ratio_truth,
    HUMAN_N_COL,
    EFFECTIVE_N_COL,
    run_odds_ratio_monte_carlo,
    summarize_monte_carlo,
)
from plotting import (
    make_monte_carlo_variance_table,
    plot_coverage,
    plot_effective_sample_size,
    plot_finite_population_coverage,
    plot_intervals,
    plot_monte_carlo_variance,
    plot_monte_carlo_variance_components,
    save_monte_carlo_variance_table,
)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [ ]:
EXAMPLE_DIR = REPO_ROOT / "Alphafold"
DATA_DIR = REPO_ROOT / "Data" / "Alphafold"
PLOTS_DIR = EXAMPLE_DIR / "plots"
RESULTS_DIR = EXAMPLE_DIR / "results"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 614
ALPHA = 0.1
TAU = 0.5
FRACS_HUMAN = np.linspace(0.02, 0.2, 20)
NUM_TRIALS = 500
TRAIN_PER_GROUP = 500
SCORE_COL = "neg_RSA_smooth30"


In [ ]:
data = pd.read_csv(DATA_DIR / "ptm_idr.csv")
analysis_df = data.dropna(subset=["p", "disordered", SCORE_COL]).copy()
analysis_df["p"] = analysis_df["p"].astype(int)
analysis_df["disordered"] = analysis_df["disordered"].astype(int)

Z = analysis_df["p"].to_numpy(dtype=bool)
Y_total = analysis_df["disordered"].to_numpy(dtype=int)
X = analysis_df[SCORE_COL].to_numpy(dtype=float)

X0, Y0_full = X[~Z], Y_total[~Z]
X1, Y1_full = X[Z], Y_total[Z]

X1_train, X1_test, Y1_train, Y1_test = train_test_split(
    X1, Y1_full, train_size=TRAIN_PER_GROUP, random_state=SEED
)
X0_train, X0_test, Y0_train, Y0_test = train_test_split(
    X0, Y0_full, train_size=TRAIN_PER_GROUP, random_state=SEED
)

X_train = np.concatenate([X1_train, X0_train])
Y_train = np.concatenate([Y1_train, Y0_train])
Z_train = np.concatenate([np.ones(len(X1_train)), np.zeros(len(X0_train))])
model = LogisticRegression(random_state=SEED, max_iter=1000).fit(
    np.column_stack([X_train, Z_train]), Y_train
)

Yhat1_test = model.predict_proba(np.column_stack([X1_test, np.ones(len(X1_test))]))[:, 1]
Yhat0_test = model.predict_proba(np.column_stack([X0_test, np.zeros(len(X0_test))]))[:, 1]
Yhat1_train = model.predict_proba(np.column_stack([X1_train, np.ones(len(X1_train))]))[:, 1]
Yhat0_train = model.predict_proba(np.column_stack([X0_train, np.zeros(len(X0_train))]))[:, 1]

true_odds_ratio, true_variance = binary_odds_ratio_truth(Y0_test, Y1_test)
mu0_pilot, mu1_pilot = float(np.mean(Y0_train)), float(np.mean(Y1_train))

def pilot_lambda(y, yhat):
    denom = np.var(yhat)
    if denom <= 1e-12:
        return 1.0
    value = np.mean((y - y.mean()) * (yhat - yhat.mean())) / denom
    return float(np.clip(value, 0, 1))

lambda0_pilot = pilot_lambda(Y0_train, Yhat0_train)
lambda1_pilot = pilot_lambda(Y1_train, Yhat1_train)
spline_score0 = np.maximum(
    Yhat0_test * (1 - 2 * lambda0_pilot * Yhat0_test) + lambda0_pilot**2 * Yhat0_test**2,
    1e-8,
)
spline_score1 = np.maximum(
    Yhat1_test * (1 - 2 * lambda1_pilot * Yhat1_test) + lambda1_pilot**2 * Yhat1_test**2,
    1e-8,
)

pd.DataFrame(
    {
        "group": ["not phosphorylated", "phosphorylated"],
        "n_test": [len(Y0_test), len(Y1_test)],
        "outcome_mean": [Y0_test.mean(), Y1_test.mean()],
        "prediction_mean": [Yhat0_test.mean(), Yhat1_test.mean()],
        "lambda_pilot": [lambda0_pilot, lambda1_pilot],
    }
)


In [ ]:
df = run_odds_ratio_monte_carlo(
    y0=Y0_test,
    yhat0=Yhat0_test,
    y1=Y1_test,
    yhat1=Yhat1_test,
    fracs_human=FRACS_HUMAN,
    alpha=ALPHA,
    num_trials=NUM_TRIALS,
    true_odds_ratio=true_odds_ratio,
    true_variance=true_variance,
    mu0_pilot=mu0_pilot,
    mu1_pilot=mu1_pilot,
    tau=TAU,
    seed=SEED,
    spline_score0=spline_score0,
    spline_score1=spline_score1,
    split_spline_budget_evenly=False,
    show_progress=True,
)

summary_df = summarize_monte_carlo(df)
mc_variance_table = make_monte_carlo_variance_table(df)

df.to_csv(RESULTS_DIR / "Alphafold_results.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "Alphafold_monte_carlo_summary.csv", index=False)
mc_variance_table.to_csv(RESULTS_DIR / "Alphafold_monte_carlo_variance_components.csv", index=False)

mc_variance_table.head(12)


In [ ]:
n_total = len(Y0_test) + len(Y1_test)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "Alphafold_effective_sample_size.png",
    n_total=n_total,
    show=False,
)
plot_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "Alphafold_coverage.png",
    n_total=n_total,
    show=False,
)
plot_finite_population_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "Alphafold_coverage_finite_population_calibrated.png",
    n_total=n_total,
    show=False,
)
plot_monte_carlo_variance(
    df,
    path=PLOTS_DIR / "Alphafold_monte_carlo_variance.png",
    n_total=n_total,
    show=False,
)
plot_monte_carlo_variance_components(
    df,
    path=PLOTS_DIR / "Alphafold_monte_carlo_variance_components.png",
    n_total=n_total,
    show=False,
)
save_monte_carlo_variance_table(
    df,
    path=PLOTS_DIR / "Alphafold_monte_carlo_variance_table.png",
    max_rows=18,
    show=False,
)
plot_intervals(
    df,
    true_value=true_odds_ratio,
    path=PLOTS_DIR / "Alphafold_intervals.png",
    estimand_label="odds ratio: phosphorylated vs not phosphorylated",
    show=False,
)
